In [0]:
bronze_path = '/Volumes/bikestore/bronze/bronze_volumes'
silver_path = '/Volumes/bikestore/silver/silver_volume'

In [0]:
bronze_tables = {
    'tmp_bronze_brands' : f'{bronze_path}/brands/',
    'tmp_bronze_categories' : f'{bronze_path}/categories/',
    'tmp_bronze_customers' : f'{bronze_path}/customers/',
    'tmp_bronze_order_items' : f'{bronze_path}/order_items',
    'tmp_bronze_orders' : f'{bronze_path}/orders/',
    'tmp_bronze_products' : f'{bronze_path}/products/',
    'tmp_bronze_staffs' : f'{bronze_path}/staffs/',
    'tmp_bronze_stocks' : f'{bronze_path}/stocks/',
    'tmp_bronze_stores' : f'{bronze_path}/stores/'
}

In [0]:
for view_name, path in bronze_tables.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name))

In [0]:
%sql
select * from tmp_bronze_customers;

In [0]:
df_product_silver = spark.sql("""
select
    product.product_id,
    product.product_name,
    product.brand_id,
    category.category_name,
    product.category_id,
    product.model_year,
    product.list_price
from tmp_bronze_products as product
left join tmp_bronze_categories as category on product.category_id = category.category_id                         
                              """"")

In [0]:
display(df_product_silver)

In [0]:
# Salvar em parquet como delta a tabela SILVER
df_product_silver.write.mode('overwrite').format('delta').option('mergeSchema', 'true').save(f'{silver_path}')